<a href="https://colab.research.google.com/github/LAtifword/latif-brain/blob/latif-brain/training/LATIF_NI_Train_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LATIF NI — real LoRA training on a free Colab GPU
Run in Google Colab with **Runtime → Change runtime type → T4 GPU** (if available). This trains **Qwen2.5-0.5B**, **not Gemini Nano**. No GPU is guaranteed; do not upload confidential datasets you lack permission to process in Colab.
Prepare a UTF-8 JSONL file with **at least 20 unique examples**: one `{ "text": "..." }` per line, each text at least 30 characters. Keep a copy offline.

In [2]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU assigned. Choose GPU runtime and reconnect.'
print('CUDA:', torch.cuda.get_device_name(0))
!git clone --depth 1 --branch latif-brain https://github.com/LAtifword/latif-brain.git /content/latif-brain
%cd /content/latif-brain
!python -m pip install -q -r training/requirements.txt 'datasets>=3,<5'


AssertionError: No CUDA GPU assigned. Choose GPU runtime and reconnect.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert len(uploaded)==1, 'Upload exactly one training JSONL file'
name = next(iter(uploaded))
Path('/content/latif-train.jsonl').write_bytes(uploaded[name])
!python training/validate_dataset.py --input /content/latif-train.jsonl --output training/data
del uploaded


In [ ]:
import subprocess, json
model='Qwen/Qwen2.5-0.5B-Instruct'
subprocess.run(['python','training/train_lora.py','--model',model,'--data','training/data/train.jsonl','--output','artifacts/lora','--steps','100','--max-length','256'],check=True)
assert Path('artifacts/lora/adapter_model.safetensors').is_file(), 'Adapter missing'
print('Training artifact exists; evaluation next.')


In [ ]:
subprocess.run(['python','training/evaluate_lora.py','--model',model,'--adapter','artifacts/lora','--data','training/data/validation.jsonl'],check=True)
print(Path('artifacts/evaluation.json').read_text())


In [ ]:
import zipfile
archive='/content/LATIF-NI-QWEN-LORA.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('artifacts/lora').rglob('*'):
        if p.is_file(): z.write(p,p.as_posix())
    z.write('artifacts/evaluation.json')
    z.write('training/data/manifest.json')
files.download(archive)
print('Download contains LoRA adapter and evaluation, NOT a standalone Android GGUF or Gemini Nano checkpoint.')
